In [ ]:
pip install pymysql

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import getpass

user = "root"
pwd = getpass.getpass("MySQL password:")
host = "localhost"
db = "sakila"

engine = create_engine(f"mysql+pymysql://{user}:{pwd}@{host}/{db}", future=True)

def run_sql(engine, sql, params=None):
    with engine.connect() as conn:
        res = conn.execute(text(sql), params or {})
        return pd.DataFrame(res.all(), columns=res.keys())

def rentals_month(engine, month: int, year: int) -> pd.DataFrame:
    sql = """
        SELECT rental_id, customer_id, rental_date, inventory_id, staff_id
        FROM rental
        WHERE MONTH(rental_date) = :m AND YEAR(rental_date) = :y
        ORDER BY customer_id, rental_date
    """
    return run_sql(engine, sql, {"m": month, "y": year})

def rental_count_month(df_rentals: pd.DataFrame, month: int, year: int) -> pd.DataFrame:
    col = f"rentals_{int(month):02d}_{int(year)}"
    counts = (
        df_rentals.groupby("customer_id", as_index=False)["rental_id"]
        .count()
        .rename(columns={"rental_id": col})
    )
    return counts

def compare_rentals(df_counts_m1: pd.DataFrame, df_counts_m2: pd.DataFrame) -> pd.DataFrame:
    merged = df_counts_m1.merge(df_counts_m2, on="customer_id", how="outer").fillna(0)
    cols = [c for c in merged.columns if c.startswith("rentals_")]
    c1, c2 = cols
    merged["difference"] = merged[c2] - merged[c1]
    merged = merged.astype({"customer_id": "int64", c1: "int64", c2: "int64", "difference": "int64"})
    return merged.sort_values(["difference", "customer_id"], ascending=[False, True]).reset_index(drop=True)

df_may = rentals_month(engine, 5, 2005)
df_june = rentals_month(engine, 6, 2005)

cnt_may = rental_count_month(df_may, 5, 2005)
cnt_june = rental_count_month(df_june, 6, 2005)

comparison = compare_rentals(cnt_may, cnt_june)
active_both = comparison[(comparison["rentals_05_2005"] > 0) & (comparison["rentals_06_2005"] > 0)]

print(comparison.head(10))
print(active_both.head(10))

   customer_id  rentals_05_2005  rentals_06_2005  difference
0           31                0               11          11
1          329                0                9           9
2          454                1               10           9
3          178                0                8           8
4          213                1                9           8
5          268                0                8           8
6          295                1                9           8
7          336                0                8           8
8          340                0                8           8
9          457                1                9           8
    customer_id  rentals_05_2005  rentals_06_2005  difference
2           454                1               10           9
4           213                1                9           8
6           295                1                9           8
9           457                1                9           8
10           27    